# Getting Started with Sandboxes

Full lifecycle: create a sandbox, run commands, write files, expose ports,
take a snapshot, stop, resume, SSH in, and clean up.

## Prerequisites
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- Sandbox SDK: `pip install azure-sandbox azure-mgmt-sandbox`

Click **Run All** or go **Step by Step**.

### 0. Initialize variables

Variables are derived from the lab folder name and your Azure subscription.
Adjust `resource_group_name` and `sandbox_group_name` if you already have resources.

In [ ]:
import os, json, subprocess
from azure.sandbox import SandboxClient
from azure.mgmt.sandbox import SandboxGroupManagementClient

# Derive names from lab folder
lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

# Get subscription from az CLI
account = json.loads(subprocess.run(
    ['az', 'account', 'show', '-o', 'json'],
    capture_output=True, text=True, check=True).stdout)

subscription_id = account['id']
resource_group_name = f'lab-{lab_name}'       # change if using existing RG
resource_group_location = 'westus2'            # change to your region
sandbox_group_name = f'{lab_name}-sg'          # change if using existing SG

print(f'Lab:            {lab_name}')
print(f'User:           {account["user"]["name"]}')
print(f'Subscription:   {account["name"]} ({subscription_id})')
print(f'Resource Group: {resource_group_name}')
print(f'Location:       {resource_group_location}')
print(f'Sandbox Group:  {sandbox_group_name}')

client = SandboxClient(subscription_id=subscription_id, resource_group=resource_group_name)
mgmt = SandboxGroupManagementClient(subscription_id=subscription_id, resource_group=resource_group_name)

### 1. Create resource group and sandbox group

In [ ]:
!az group create --name {resource_group_name} --location {resource_group_location} -o none

group = mgmt.create_group(sandbox_group_name, location=resource_group_location)
print(f'Sandbox group: {group["name"]} ({group["location"]})')

### 2. Create a sandbox

In [ ]:
sbx = client.create_sandbox(sandbox_group_name, disk='ubuntu')
sandbox_id = sbx['id']
print(f'Sandbox: {sandbox_id}  state={sbx["state"]}')

### 3. Execute commands

In [ ]:
result = client.exec(sandbox_id, sandbox_group_name, 'echo Hello from sandbox && whoami && uname -a')
print(f'exitCode: {result["exitCode"]}')
print(result['stdout'])

### 4. Write and read files

In [ ]:
client.write_file(sandbox_id, sandbox_group_name, '/tmp/hello.txt', 'Hello from SDK!')
content = client.read_file(sandbox_id, sandbox_group_name, '/tmp/hello.txt')
print(f'File content: {content.decode()}')

### 5. Add a port

In [ ]:
ports = client.add_port(sandbox_id, sandbox_group_name, 8080, anonymous=True)
for p in ports.get('ports', []):
    print(f'port={p["port"]}  url={p.get("url", "n/a")}')

### 6. Stats + Snapshot

In [ ]:
stats = client.get_stats(sandbox_id, sandbox_group_name)
mem = stats.get('memory', {})
print(f'Memory: {mem.get("usedBytes",0)//1024//1024}MB / {mem.get("totalBytes",0)//1024//1024}MB')

snap = client.create_snapshot(sandbox_id, sandbox_group_name, name='getting-started')
print(f'Snapshot: {snap.get("id", "n/a")}')

### 7. Stop and resume

In [ ]:
import time
client.stop_sandbox(sandbox_id, sandbox_group_name)
print('Stopped')
time.sleep(3)

client.resume_sandbox(sandbox_id, sandbox_group_name)
time.sleep(5)
result = client.exec(sandbox_id, sandbox_group_name, 'echo Back from suspend!')
print(result['stdout'].strip())

### 8. SSH into the sandbox

Jupyter can't run interactive terminal sessions, but you can SSH from
VS Code's integrated terminal. Run the command below:

**How SSH works:**
- No port 22, no SSH keys, no SSH daemon
- WebSocket connection to the sandbox management API
- Authenticated via your `az login` token
- Full TTY: colors, tab completion, vim, tmux all work

**Tips for long-running sessions:**
```bash
# Inside the sandbox, use tmux to keep sessions alive:
tmux new -s work           # start a named session
# ... do your work ...
# Ctrl+B, D               # detach (session keeps running)

# Later, SSH back in and reattach:
tmux attach -t work        # pick up exactly where you left off
tmux ls                    # list all sessions

# Useful tmux commands:
# Ctrl+B, C               # new window
# Ctrl+B, N               # next window
# Ctrl+B, %               # split pane vertically
# Ctrl+B, "               # split pane horizontally
```

In [ ]:
# Copy-paste this into VS Code's integrated terminal (Ctrl+`)
print('SSH into your sandbox:')
print(f'  node plugin/skills/azure-sandbox/assets/ssh.mjs {sandbox_id} -g {resource_group_name} -s {sandbox_group_name}')
print(f'  az sandbox ssh -g {resource_group_name} -s {sandbox_group_name} --id {sandbox_id}')
print()
print('Press Ctrl+D to exit SSH.')
print()
print('Tip: use tmux inside the sandbox for persistent sessions:')
print('  tmux new -s work     # start')
print('  Ctrl+B, D            # detach')
print('  tmux attach -t work  # reattach later')

### 9. Clean up

In [ ]:
client.delete_sandbox(sandbox_id, sandbox_group_name)
print(f'Deleted sandbox: {sandbox_id}')

mgmt.delete_group(sandbox_group_name)
print(f'Deleted group: {sandbox_group_name}')

!az group delete --name {resource_group_name} --yes --no-wait
print(f'Deleting resource group: {resource_group_name}')